# 08 — FastAPI Serving Demo

Test the FastAPI endpoint from the notebook:
- Start the server in a background process
- Send POST requests with `requests`
- Parse and visualize the 24-hour forecast with uncertainty bands

**Prerequisites:** Model trained (NB05) + Feast materialized (NB04).

Alternatively, start the server manually:
```bash
uvicorn serving.api:app --reload --port 8000
```

In [ ]:
import sys
sys.path.insert(0, '..')

import subprocess
import time
import requests
import json
import pandas as pd
import plotly.graph_objects as go

## 8.1 Start the API server (background)

In [ ]:
import os

# Start server as background process
server = subprocess.Popen(
    ['uvicorn', 'serving.api:app', '--port', '8000'],
    cwd='..',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(3)  # wait for startup

# Health check
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print('Health:', r.json())
except Exception as e:
    print(f'Server not ready: {e}. Try starting manually: uvicorn serving.api:app --port 8000')

## 8.2 Model info

In [ ]:
r = requests.get('http://localhost:8000/model-info')
info = r.json()
print(f"Model type: {info['model_type']}")
print(f"Horizons: {info['horizons']}")
print(f"Features: {len(info['feature_columns'])}")
print(f"Quantile models loaded: {info['quantile_models_loaded']}")

## 8.3 Forecast request

In [ ]:
payload = {
    'market_zone': 'DE_LU',
    'prediction_ts_utc': '2024-01-15T12:00:00Z'
}

r = requests.post('http://localhost:8000/predict', json=payload)
response = r.json()
print(f'Status: {r.status_code}')
print(f'Zone: {response["market_zone"]}')
print(f'From: {response["prediction_ts_utc"]}')
print(f'Horizons returned: {len(response["forecasts"])}')

## 8.4 Visualize 24-hour forecast with uncertainty bands

In [ ]:
forecasts = response['forecasts']
horizons     = [f['horizon'] for f in forecasts]
mean_prices  = [f['price_eur_mwh'] for f in forecasts]
lower_prices = [f.get('lower_bound') for f in forecasts]
upper_prices = [f.get('upper_bound') for f in forecasts]

fig = go.Figure()

if lower_prices[0] is not None:
    fig.add_trace(go.Scatter(
        x=horizons + horizons[::-1],
        y=upper_prices + lower_prices[::-1],
        fill='toself', fillcolor='rgba(99,110,250,0.15)',
        line=dict(color='rgba(0,0,0,0)'), name='80% prediction interval'
    ))

fig.add_trace(go.Scatter(
    x=horizons, y=mean_prices,
    mode='lines+markers',
    name='Forecast (mean)',
    line=dict(color='#636EFA', width=2.5),
    marker=dict(size=7),
))

fig.update_layout(
    title=f'24-Hour Price Forecast from {payload["prediction_ts_utc"]} — DE_LU',
    xaxis_title='Horizon', yaxis_title='EUR/MWh',
    template='plotly_dark', height=420
)
fig.show()

## 8.5 Cleanup — stop server

In [ ]:
server.terminate()
print('Server stopped.')